# Talent Fit Scoring Pipeline
Rank candidates by fit score using job title similarity and connection count.

In [3]:
from pathlib import Path
from config import DATA_FILE
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.feature_engineering import add_features
from src.ranking import rank_candidates
from src.reranking import rerank
from src.evaluation import ndcg_at_k

try:
    ROOT = Path(__file__).resolve().parent  # running as a script
except NameError:
    ROOT = Path().resolve()                 # running as a notebook
df = load_data(ROOT / DATA_FILE)
df.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


## Inspect raw data

In [4]:
print(df.shape)
print(df.dtypes)
df["connection"].value_counts()

(104, 5)
id              int64
job_title      object
location       object
connection     object
fit           float64
dtype: object


connection
500+     44
85        7
61        7
44        6
1         5
2         4
390       2
57        2
7         2
4         2
82        1
16        1
5         1
409       1
52        1
455       1
212       1
174       1
268       1
50        1
40        1
18        1
349       1
155       1
39        1
64        1
9         1
415       1
19        1
71        1
48        1
103       1
49        1
Name: count, dtype: int64

## Preprocess

In [5]:
df = preprocess(df)
# connections_norm: LinkedIn connections is a proxy for how networked/active someone is in their field.
# A well-connected person is likely more experienced or engaged professionally.
# Raw values go from 0 to "500+" so we normalize (divide by 500) to get a 0-1 scale.
# Without this, we can't combine it with fit (also 0-1) — they'd be on completely different
# scales and connections would dominate unfairly.
df[["job_title", "job_title_clean", "connections_raw", "connections_norm"]].head(10)

,job_title,job_title_clean,connections_raw,connections_norm
0,2019 C.T. Bauer College of Business Graduate (...,bauer college business graduate magna cum laud...,85,0.170
1,Native English Teacher at EPIK (English Progra...,native english teacher epik english program korea,500,1.000
2,Aspiring Human Resources Professional,aspiring human resource professional,44,0.088
3,People Development Coordinator at Ryan,people development coordinator ryan,500,1.000
4,Advisory Board Member at Celal Bayar University,advisory board member celal bayar university,500,1.000
5,Aspiring Human Resources Specialist,aspiring human resource specialist,1,0.002
6,Student at Humber College and Aspiring Human R...,student humber college aspiring human resource...,61,0.122
7,HR Senior Specialist,senior specialist,500,1.000
8,Student at Humber College and Aspiring Human R...,student humber college aspiring human resource...,61,0.122
9,Seeking Human Resources HRIS and Generalist Po...,seeking human resource hris generalist position,500,1.000


## Feature engineering (TF-IDF cosine similarity)

In [6]:
df = add_features(df)
df[["job_title_clean", "fit"]].sort_values("fit", ascending=False).head(10)

,job_title_clean,fit
16,aspiring human resource professional,0.922173
2,aspiring human resource professional,0.922173
20,aspiring human resource professional,0.922173
32,aspiring human resource professional,0.922173
57,aspiring human resource professional,0.922173
96,aspiring human resource professional,0.922173
45,aspiring human resource professional,0.922173
23,aspiring human resource specialist,0.913002
59,aspiring human resource specialist,0.913002
5,aspiring human resource specialist,0.913002


## Rank candidates

In [7]:
ranked = rank_candidates(df)
ranked[["id", "job_title", "location", "connections_raw", "fit"]].head(10)

,id,job_title,location,connections_raw,fit
0,28,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.845277
1,30,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.845277
2,40,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.844399
3,10,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.844399
4,62,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.844399
5,53,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.844399
6,27,Aspiring Human Resources Management student se...,"Houston, Texas Area",500,0.801423
7,29,Aspiring Human Resources Management student se...,"Houston, Texas Area",500,0.801423
8,67,"Human Resources, Staffing and Recruiting Profe...","Jackson, Mississippi Area",500,0.785814
9,69,"Director of Human Resources North America, Gro...","Greater Grand Rapids, Michigan Area",500,0.716472


## Re-rank after human feedback

In [8]:
# Full reviewed list (43 candidates) — kept for reference:
# STARRED_IDS = [1, 3, 6, 7, 9, 10, 14, 15, 17, 19, 21, 24, 25, 27, 28, 29, 30, 31, 33, 36, 37, 39, 40, 44, 46, 49, 50, 52, 53, 57, 58, 60, 62, 66, 72, 73, 75, 76, 79, 82, 97, 99, 100]
STARRED_IDS = [3, 6, 17, 19, 31, 100]

reranked = rerank(ranked, STARRED_IDS)
reranked[["id", "job_title", "location", "connections_raw", "fit"]].head(10)

,id,job_title,location,connections_raw,fit
0,97,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.793262
1,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.791642
2,17,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.791642
3,33,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.791642
4,58,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.791642
5,46,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.791642
6,21,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.791642
7,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.786263
8,36,Aspiring Human Resources Specialist,Greater New York City Area,1,0.786263
9,49,Aspiring Human Resources Specialist,Greater New York City Area,1,0.786263


## Hyperparameter tuning
Grid search over w (title weight) and alpha (re-rank blend) to maximise NDCG@10.
df_feats is saved before ranking so we can restart cleanly on each iteration.

In [9]:
import numpy as np

df_feats = df.copy()  # snapshot after add_features, before any ranking modifies fit

best_ndcg, best_w, best_alpha = -1, None, None
results = []

for w in np.arange(0.1, 1.0, 0.1):
    for alpha in np.arange(0.1, 1.0, 0.1):
        r  = rank_candidates(df_feats, w=round(w, 1))
        rr = rerank(r, STARRED_IDS, alpha=round(alpha, 1))
        score = ndcg_at_k(rr, STARRED_IDS)
        results.append((round(w, 1), round(alpha, 1), round(score, 4)))
        if score > best_ndcg:
            best_ndcg, best_w, best_alpha = score, round(w, 1), round(alpha, 1)

print(f"Best w={best_w}, alpha={best_alpha} → NDCG@10={best_ndcg:.4f}")

Best w=0.7, alpha=0.6 → NDCG@10=0.4377


## Evaluate

In [10]:
# @10 is the standard in information retrieval — users realistically only look at the first page of results.
print(f"NDCG@10 before re-ranking: {ndcg_at_k(ranked,   STARRED_IDS):.4f}")
print(f"NDCG@10 after  re-ranking: {ndcg_at_k(reranked, STARRED_IDS):.4f}")

NDCG@10 before re-ranking: 0.0000
NDCG@10 after  re-ranking: 0.4377
